Reload Chroma ➜ contextualize follow-up questions with chat history (ChatAnthropic) ➜ retrieve with the standalone question

In [4]:
# reload vector store
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

vectorstore = Chroma(
    persist_directory=".chroma/student-rag",
    embedding_function=embeddings,
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

print(f"{vectorstore._collection.count()} vectors loaded")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14992.65it/s]


549 vectors loaded


In [8]:
# chat model (ChatAnthropic)
from dotenv import load_dotenv
from langchain_anthropic import ChatAnthropic

load_dotenv()  # loads ANTHROPIC_API_KEY from .env

llm = ChatAnthropic(
    model="claude-opus-5-5",
    max_tokens=4096,
    effort="low",
)

In [9]:
# contextualize question: {"question", "chat_history"} ➜ standalone question ➜ retriever
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

contextualize_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Given a chat history and the latest user question, which might reference context in the "
     "chat history, rewrite it as a standalone question that can be understood without the chat "
     "history. Do NOT answer the question. If it is already standalone, return it unchanged. "
     "Reply with the question only."),
    MessagesPlaceholder("chat_history"),  # list[HumanMessage | AIMessage]
    ("human", "{question}"),
])

contextualize_question = contextualize_prompt | llm | StrOutputParser()

history_aware_retriever = contextualize_question | retriever  # returns list[Document]

In [10]:
# Build a contextualize-question step that uses chat history to rewrite follow-ups.
from langchain_core.messages import HumanMessage, AIMessage

chat_history = [
    HumanMessage("What is the Transformer architecture?"),
    AIMessage("The Transformer is a sequence model built entirely on attention, with no recurrence or convolutions."),
]
follow_up = "How many attention heads does it use?"

standalone = contextualize_question.invoke({"question": follow_up, "chat_history": chat_history})
print(f"follow-up:  {follow_up}\nstandalone: {standalone}\n")

for doc in retriever.invoke(standalone):
    print(doc.metadata["source"], "p.", doc.metadata.get("page"))

AnthropicInvalidRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is too low to access the Anthropic API. Please go to Plans & Billing to upgrade or purchase credits.'}, 'request_id': 'req_011CgBVas5bHEooZ6EvhUxAd'}